# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
#%pip install datasets
from datasets import load_dataset
ds1 = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
row1 = next(iter(ds1))
print(row1)
ds2 = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", streaming=True, split="train")
row2 = next(iter(ds2))
print(row2)
ds3 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")
row3 = next(iter(ds3))
print(row3)
ds4 = load_dataset("FlyRank/internship-warehouse", "dim_clients", streaming=True, split="train")
row4 = next(iter(ds4))
print(row4)
# fact_content_daily_performance: one row = one webpage on one day. The date ranges from Jan 27, 2025 to June 30, 2026 on the dataset.
# fact_content_query_90d: one row = one webpage over a 90-day period for a specific query.
# dim_content: one row = one webpage.
# dim_clients: one row = one client.

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}
{'client_hash_id': 'client_08a6a72ff48e62c0', 'content_hash_id': 'content_447894f2faf0d2bc', 'query_hash_id': 'query_58b1b001f839d699', 'query_char_count': 17, 'query_token_count': 3, 'window_start': datetime.date(2026, 4, 2), 'window_end': datetime.date(2026, 6, 30), 'impress

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Context: report_date, client_hash_id, and content_hash_id
# Feature: ga4_pageviews, sessions_organic, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, and scroll_events
# Feature and label: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_users, ga4_engaged_sessions, and ga4_total_engagement_sec because these are the important metrics for measuring success
# Excluded: client_has_gsc, client_has_ga4, gsc_data_available, and ga4_data_available because these don't give meaningful information in predicting future success of a page, and gsc_sum_position because it could be calculated as gsc_impressions * gsc_avg_position 

"\nfact_content_daily_performance table:\nContext: report_date, client_hash_id, and content_hash_id\nFeature: ga4_pageviews, sessions_organic, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, and scroll_events\nFeature and label: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_users, ga4_engaged_sessions, and ga4_total_engagement_sec because these are the important metrics for measuring success\nExcluded: client_has_gsc, client_has_ga4, gsc_data_available, and ga4_data_available because these don't give meaningful information in predicting future success of a page, and gsc_sum_position because it could be calculated as gsc_impressions * gsc_avg_position \n"

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
it1 = iter(ds1)
for i in range(3):
    row = next(it1)
    print(row['gsc_sum_position'], row['gsc_avg_position'] * row['gsc_impressions'])
    print(row)
print('The values in the report_date (Jan 1, 2025) and content_hash_id verifies that one row = one webpage on one day.')
print('gsc_sum_position = gsc_avg_position * gsc_impressions, which verifies that gsc_sum_position could be excluded.')
it2 = iter(ds2)
for i in range(3):
    row = next(it2)
    print(row)
print('The values in the content_hash_id, window_start (April 2, 2026), and window_end (June 30, 2026) verifies that one row = one webpage over a 90 day period (29 in April + 31 in May + 30 in June).')
it3 = iter(ds3)
for i in range(3):
    row = next(it3)
    print(row)
print('The values in content_hash_id verifies that one row = one webpage. The other keys like url_char_count and content_created_date are true about a webpage rather than a specific date.')
it4 = iter(ds4)
for i in range(3):
    row = next(it4)
    print(row)
print('It has client_hash_id, but not content_hash_id, which verifies that one row = one client.')

115 115.0
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}
358 358.0
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_fe8e8155ce1d47a2', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# What the data never tell: causal relationship like increase in variable A causes variable B to decrease.
# Unbalanced history: some pages will have data for a much longer period than others, so not every page provides an equal amount of data for training the model.
# GSC-only early rows: if a page does not have GA4 for the early rows, the early rows should be removed so that only rows with both GSC and GA4 is used for training.
# Window overlap: if a 90 day window for a page starts after the start date but before the end date of another 90 day window for the same page, using one for training and other for testing would cause data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.